# 12 — Interrupt, resume, compact: the ops story

**What you'll learn**

- Build a `DurableRunner` around the chapter-02 loop that checkpoints `{messages, step, meta}` after every step with `shoplab.controls.Checkpoint`
- Survive a mid-run crash: catch an interrupt, then confirm the last checkpoint is intact on disk with no half-finished side effects
- Resume a killed triage from its checkpoint and reach the same decision as an uninterrupted run — without re-paying for the lookups already done
- Compact a long session's finished history with `shoplab.context.compact` before resuming, so the continued run fits the context window
- Map each real interrupt — crash, rate limit, human stop, deploy, overflow — to what survives and how you resume

*Time: ~3 min on a first live run; under a minute cached. Cost: ~$0.01. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The frozen tracing cell from notebook 03. Leave it in: the checkpoint, crash, and resume below each show up as their own runs, so you can watch a triage stop on one process and pick back up on another.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The run that outlives its process

Every agent run so far has been a process: a Python list of messages held in one kernel's memory, growing until the loop stops. Processes die. The box reboots for a kernel upgrade, the rate limiter returns a 429, someone hits Ctrl-C, a deploy rolls the fleet mid-ticket. When the run's entire state lives in the memory of the process that dies, the death is total: the ticket starts over from zero, re-paying for every lookup and every token it had already spent.

Durable execution is the discipline that fixes this — keep the run's state *outside* the process, on disk or in a store, updated often enough that a dead process loses at most one step. You already built both halves. Chapter 08 gave you `Checkpoint`: state you can save and load. Chapter 10 gave you `compact`: state you can shrink. This chapter wires them into the one operational story every long-running agent needs — interrupt, resume, compact — and nothing here is a new primitive, only the two you have, used together.

## The triage we refuse to lose

We reuse the WORLD's vip opened-return ticket, `TKT-2228` — the same one chapter 08 moved money on. Run it once, uninterrupted, through the plain chapter-02 loop — and note that durability wraps *any* loop, the chapter-11 deep agent included; we use the plainest one here so the checkpoint machinery, not the agent, stays in focus. That decision is our reference: every version below, after being crashed, resumed, and compacted, must still reach it.

In [ ]:
import json
from pathlib import Path

import shoplab.llm
from shoplab.loop import run_agent
from shoplab.tools import standard_tools, Ledger, run_tool, to_openai_tools
from shoplab.controls import Checkpoint
from shoplab.context import compact, count_tokens
from shoplab.world import load_tickets

TICKETS = {t["ticket_id"]: t for split in load_tickets().values() for t in split}

SYSTEM = ("You are the operations desk agent for Larkspur Outfitters. Look up the "
          "order, the customer, and the one relevant policy, then stop looking "
          "things up. Then carry out the decision: call issue_refund when money is "
          "due, create_replacement for a replacement, or escalate for a human "
          "review; only then call finish with decision, policy_id, and refund_usd "
          "(number or null). Decisions follow shop policy, not sympathy.")

def render_ticket(t):                            # the renderer from chapter 04
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']}, qty {t['qty']}, condition "
            f"{t['item_condition']}, days since delivery {t['days_since_delivery']}, "
            f"photo evidence {t['evidence_photo']}, requested action "
            f"{t['requested_action']}. Customer writes: {t['reason_text']}")

ticket = TICKETS["TKT-2228"]                     # CUST-01 (vip), opened, in-window refund
base = run_agent(render_ticket(ticket), standard_tools(), system=SYSTEM, max_steps=10)
print("uninterrupted decision:", base.answer)

> **What you should see:** the loop works the ticket end to end and lands on `approve_refund` for the vip's opened, in-window return — `policy_id` `pol-restocking`, `refund_usd` $129.99 (the full item value; the restocking fee is waived for a vip). This is the answer every version below must still reach after being interrupted. Right now it lives only in this kernel's memory: kill the process here and the whole triage is gone, with nothing on disk to pick up from.

## A runner that saves after every step

`run_agent` keeps the whole run in one Python list that vanishes with the process. The durable version is the same loop with one addition: after every step — the model call, its tool calls, their results — write the running state to disk with chapter 08's `Checkpoint.save`. A *resume* is then nothing more than loading that state and continuing the loop from the step it reached.

We factor the loop into a plain `drive` function — the resumable core, checkpointing each step — and wrap it in a `DurableRunner` for ergonomics. `Interrupted` is our stand-in for a crash: raised mid-run to prove the checkpoint written just before it survives. Note what `drive` reuses — `shoplab.llm.complete`, `run_tool`, `to_openai_tools`, `Checkpoint` — all already built; the only new idea is the save-point.

In [ ]:
class Interrupted(RuntimeError):
    """Stand-in for a crash: raised mid-run to prove the checkpoint outlives it."""

def drive(messages, step, tools, path, *, model=None, max_steps=10, crash_at=None):
    answer, stop = None, "max_steps"
    while step < max_steps:
        step += 1
        msg = shoplab.llm.complete(messages, tools=to_openai_tools(tools),
                                   model=model).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls or []:
            out = run_tool(tools, tc.function.name, json.loads(tc.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out[:2000]})
            if tc.function.name == "finish":
                answer, stop = json.loads(tc.function.arguments), "finish"
        Checkpoint.save({"messages": messages, "step": step, "meta": {"stop": stop}}, path)
        if stop != "max_steps":
            break
        if crash_at is not None and step == crash_at:
            raise Interrupted(f"process killed after step {step}")
    return {"answer": answer, "step": step, "stop": stop}

In [ ]:
class DurableRunner:
    """run_agent's loop with a save-point after every step; resumable from disk."""
    def __init__(self, path, tools, *, system=None, model=None, max_steps=10):
        self.path, self.tools, self.system = Path(path), tools, system
        self.model, self.max_steps = model, max_steps

    def start(self, task, *, crash_at=None):
        msgs = ([{"role": "system", "content": self.system}] if self.system else [])
        msgs.append({"role": "user", "content": task})
        return drive(msgs, 0, self.tools, self.path, model=self.model,
                     max_steps=self.max_steps, crash_at=crash_at)

    def resume(self, *, crash_at=None):
        cp = Checkpoint.load(self.path)
        return drive(cp["messages"], cp["step"], self.tools, self.path,
                     model=self.model, max_steps=self.max_steps, crash_at=crash_at)

Point it at the reference ticket and let it run to the end. It should reach the same decision as the uninterrupted run — and, the new part, leave a checkpoint on disk holding the full message list and the step it reached. That file is the run's memory, now living outside the process.

In [ ]:
led = Ledger()
runner = DurableRunner("artifacts/checkpoints/triage.json",
                       standard_tools(led), system=SYSTEM)

n0 = len(shoplab.llm.LEDGER)
full = runner.start(render_ticket(ticket))
full_calls = len(shoplab.llm.LEDGER) - n0

saved = Checkpoint.load("artifacts/checkpoints/triage.json")
print("durable full run ->", full["answer"])
print(f"reached step {full['step']} in {full_calls} model calls; ledger {led.entries}")
print(f"checkpoint on disk: step {saved['step']}, {len(saved['messages'])} messages")

> **What you should see:** the durable run reaches the same `approve_refund` / $129.99, moves the refund into the ledger, and now a checkpoint file sits under `artifacts/checkpoints/` holding every message and the final step number. Nothing about the answer changed; what changed is that the run's state is recoverable. Had this kernel died on the last step, the next process could read that file and know exactly where the triage stood.

## Interrupt: the process dies mid-ticket

Now kill it mid-run. `crash_at=2` raises `Interrupted` right after the second step's checkpoint is written — the model has looked up the order and the customer, but has not yet decided anything or moved any money. In a real deployment this is the OOM kill, the 429, the Ctrl-C, the rolling restart. Catch the exception the way a supervisor catches a dead worker, then look at what is left on disk.

In [ ]:
led = Ledger()
crashed = DurableRunner("artifacts/checkpoints/crashed.json",
                        standard_tools(led), system=SYSTEM)
try:
    crashed.start(render_ticket(ticket), crash_at=2)
except Interrupted as stop:
    print("crash:", stop)

cp = Checkpoint.load("artifacts/checkpoints/crashed.json")
print(f"survived on disk: step {cp['step']}, {len(cp['messages'])} messages")
print("side effects so far:", led.entries)

> **What you should see:** the run raises `Interrupted` after step 2, but the checkpoint written just before the crash is intact — step 2, with the order and customer lookups already in its messages — and the ledger is empty: no refund was issued, because the crash landed before the decision. The process is gone; the progress is not. That is the whole promise of durable execution: a crash costs the step in flight, not the run.

## Resume: continue, do not restart

A restart would re-run the two lookups the crashed process already paid for. A resume does not. Load the checkpoint and continue the loop from step 3 — the order and customer are already sitting in the messages, so the model reads them back and moves straight to the policy and the refund. Same ticket, same decision, but only the work that was actually left.

In [ ]:
led = Ledger()
n1 = len(shoplab.llm.LEDGER)
resumed = DurableRunner("artifacts/checkpoints/crashed.json",
                        standard_tools(led), system=SYSTEM).resume()
resume_calls = len(shoplab.llm.LEDGER) - n1

print("resumed decision:     ", resumed["answer"])
print("uninterrupted decision:", base.answer)
print(f"resume spent {resume_calls} model calls; a cold restart spent {full_calls}")
print("side effects:", led.entries)

> **What you should see:** the resumed run reaches the same decision as the uninterrupted reference — `approve_refund`, `pol-restocking`, $129.99 — and it spends only the calls for the steps that were left, fewer than the cold run's total. The two lookups from before the crash were never redone; they were read back from the checkpoint. That gap between resume-cost and cold-start-cost is the entire economic argument for checkpointing a long run. (The refund fires on resume — a real deployment must make risky tools idempotent so a resumed step cannot pay twice; exercise 2.)

### The window this demo cannot show

Look at the order of operations inside `drive`: `run_tool` executes the side effect, and only afterwards does `Checkpoint.save` record that the step happened. A real crash between those two lines leaves durable state one step behind reality -- the resumed run re-asks the model, the model reasonably re-issues the refund, and the customer is paid twice. Our `crash_at` fires after the save, so this demo can never expose that window; do not mistake a passing demo for a closed gap.

Production systems close it in one of two ways. An **idempotency key** derives the write's identity from its content (`ticket_id + policy_id + amount`), so the downstream system treats a replay as the same refund and deduplicates it. A **write-ahead journal** records the intent before executing (`pending`), marks it `done` after, and reconciles on resume: a `pending` entry with no `done` is the only case that needs human eyes. Either way the invariant is the same -- a risky write and the record of that write must not be separable by a crash.

## Compact before you resume

An ops desk does not run one ticket and stop; it works ticket after ticket in one long-lived session, and the checkpoint grows without bound. Resume a session whose history has outgrown the context window and the model's next call overflows before it starts. The fix is chapter 10's `compact`: fold the finished, dead-weight history into one summary line and keep only what the next task needs. This is MemGPT's idea exactly — treat the window like RAM and page the cold pages out to external storage ([MemGPT, arXiv:2310.08560](https://arxiv.org/abs/2310.08560)).

Here the finished ticket is cold. A fresh ticket, `TKT-2226`, arrives in the same session; we append it, compact the completed triage that precedes it, and resume from the smaller checkpoint. We reset the compacted checkpoint's `step` to 0 — the folded session is a fresh leg with its own step budget.

In [ ]:
ticket_b = TICKETS["TKT-2226"]                   # CUST-02 (vip), unopened, 55 days -> out of window
session = Checkpoint.load("artifacts/checkpoints/triage.json")
session["messages"].append({"role": "user", "content": render_ticket(ticket_b)})
before = count_tokens(session["messages"])

folded = compact(session["messages"], keep_last=1)
after = count_tokens(folded)
Checkpoint.save({"messages": folded, "step": 0, "meta": {}}, "artifacts/checkpoints/session.json")
print(f"session carried {len(session['messages'])} messages, {before} tokens")
print(f"compacted to    {len(folded)} messages, {after} tokens")

done = DurableRunner("artifacts/checkpoints/session.json",
                     standard_tools(), system=SYSTEM).resume()
print("resumed on the new ticket ->", done["answer"])

> **What you should see:** the session carried the whole finished ticket-A transcript plus the new ticket, and `compact` folds the completed history down to a single summary line — the token count drops sharply, here by roughly two-thirds. Resuming from the compacted checkpoint, the desk triages the new ticket correctly — an out-of-window `deny` on `pol-returns` — because the facts it needs are fetched fresh, not read from the folded summary. Compact the history you are done with; never compact the facts the live task still needs — that is the lossy edge chapter 10 warned about.

## The ops story in one table

Every interrupt has the same shape: something outside the agent stops the process, and durable state decides whether that costs one step or the whole run. What changes across causes is what you do on the way back in.

| Interrupt cause | What survives | How you resume |
|---|---|---|
| Process crash / OOM kill | the last per-step checkpoint on disk | load it, `drive` from `step + 1` |
| Rate limit (HTTP 429) | the checkpoint; the failed call was never applied | resume; the next call retries the same step |
| Human stop / Ctrl-C | the checkpoint at the last completed step | resume when a person is ready, or hand it to a reviewer |
| Deploy / rolling restart | the checkpoint in shared storage, not process memory | a new process loads the file and continues |
| Context overflow | the checkpoint, once `compact`ed to fit | resume from the compacted state |

## Where durable state comes from in production

We hand-rolled the durable layer — `Checkpoint.save` plus a `drive` loop that calls it — to show there is no magic in it: a run is resumable exactly when its state is on disk in a form you can load back. Production frameworks give you this as a first-class layer instead of a hand-rolled one. LangGraph's persistence is built on checkpointers (short-term memory) and stores (long-term memory): the runtime checkpoints the graph's state at each step, so a run resumes from the last checkpoint after a crash or a human-in-the-loop pause ([LangChain docs](https://docs.langchain.com/oss/python/langgraph/persistence)). Chapter 16 wires one of those checkpointers to this same ops desk; the mechanics you built here are exactly what it automates.

## Recap

| Concept | One-liner |
|---|---|
| Durable execution | keep run state outside the process, so a crash costs a step, not the whole run. |
| `Checkpoint` (ch08) | `{messages, step, meta}` written atomically — the unit of durable state, imported not rebuilt. |
| `drive` | the resumable core: continue the loop from `(messages, step)`, checkpointing after every step. |
| `DurableRunner` | `run_agent`'s loop plus a save-point; `start` a fresh task or `resume` one from disk. |
| Interrupt | a crash mid-run is just a raised exception; the checkpoint written first outlives it. |
| Resume | load the checkpoint and continue — never re-pay for the lookups already on disk. |
| Idempotency | resume re-runs whatever step was in flight, so risky tools must not double-fire. |
| Compact then resume | fold the finished history before continuing so a long session fits the window (MemGPT paging). |
| Ops table | crash, rate limit, human stop, deploy, overflow — one durable-state answer each. |
| Frameworks | LangGraph checkpointers make this a first-class persistence layer (chapter 16). |

## Exercises

1. Checkpoint every 2 steps and resume from the earlier one. Change `drive` to save only on even steps (or write numbered files `step-02.json`, `step-04.json`, ...), interrupt at step 4, then resume from the step-2 checkpoint rather than the latest. How many calls does the coarser cadence re-pay compared with per-step checkpointing, and what is the tradeoff between save frequency and wasted work on resume?
2. Close the window from the section above: give `drive` a write-ahead journal (`journal.jsonl` next to the checkpoint) that records `pending` before `run_tool` executes `issue_refund` and `done` after, then make `resume` refuse to continue while a `pending` entry has no `done`. Crash between the two records by hand and watch it catch you.
3. Compact-then-resume a 12-step run and compare token totals. Chain several tickets through one `DurableRunner` until the session is long enough to need compaction, then resume it two ways: cold (full history) and compacted (tune `keep_last`). Tabulate the prompt tokens each resume sends on its next call, and confirm the compacted resume still reaches the right decision on the live ticket — the point where compaction stops being free. (Cheap: a dozen steps is well under a cent.)

**Next up:** Part 4 opens the ops desk to the outside world — interoperability. Chapter 13 exposes the shop's tools over the Model Context Protocol (MCP) so any client can call them, and chapter 14 lets separate agents talk to each other over A2A. The durable, resumable loop you built here is the thing those protocols will drive.